# 02 — Data Preprocessing & Cleaning
**Goal:** Clean raw data, filter specialties, encode labels, split and save  
**Key decisions:**  
- Keep only 8 true medical specialties (remove format-based labels)  
- Remove transcriptions with fewer than 10 words  
- Undersample Surgery to reduce class imbalance  
- Stratified 80/20 train/test split

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('/kaggle/input/mtsamples/mtsamples.csv')

df['medical_specialty'] = df['medical_specialty'].str.strip()

print("Raw data shape:", df.shape)

In [ ]:
# Keep True Medical Specialties Only
# Removed: Consult-History, SOAP Notes, General Medicine, Office Notes
# Reason: These are document formats, not specialties.
# They share language with all other specialties causing label ambiguity.

true_specialties = [
    'Surgery',
    'Cardiovascular / Pulmonary',
    'Orthopedic',
    'Radiology',
    'Neurology',
    'Gastroenterology',
    'Urology',
    'Obstetrics / Gynecology'
]

df = df[df['medical_specialty'].isin(true_specialties)].copy()

print("After specialty filter:", df.shape)
print(df['medical_specialty'].value_counts())

In [ ]:
# Drop Nulls & Short Texts

df = df.dropna(subset=['transcription'])
df['text_length'] = df['transcription'].apply(lambda x: len(str(x).split()))
df = df[df['text_length'] >= 10].reset_index(drop=True)

print("After null/short text removal:", df.shape)

In [ ]:
# Undersample Surgery
# Surgery has 1103 samples vs ~160-370 for others
# Undersample to 400 to reduce class imbalance

surgery_df = df[df['medical_specialty'] == 'Surgery'].sample(400, random_state=42)
other_df = df[df['medical_specialty'] != 'Surgery']
df = pd.concat([surgery_df, other_df]).reset_index(drop=True)

print("After undersampling Surgery:")
print(df['medical_specialty'].value_counts())
print("\nTotal samples:", len(df))

In [ ]:
# Label Encoding
le = LabelEncoder()
df['label'] = le.fit_transform(df['medical_specialty'])


# Save label mapping
label_mapping = pd.DataFrame({
    'specialty': le.classes_,
    'label': range(len(le.classes_))
})

label_mapping.to_csv('label_mapping.csv', index=False)

print("Label Mapping:")
print(label_mapping)

In [ ]:
# Save Label Encoder

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
    
print("Label encoder saved as label_encoder.pkl")

In [ ]:
# Stratified Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    df['transcription'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']    # ensures all classes in both splits
)

print(f"Train size : {len(X_train)}")
print(f"Test size  : {len(X_test)}")
print(f"Classes in train: {len(y_train.unique())}")
print(f"Classes in test : {len(y_test.unique())}")

In [ ]:
# Save Cleaned Data

df.to_csv('mtsamples_cleaned.csv', index=False)

print("Saved: mtsamples_cleaned.csv")